# Semantic Knowledge Base

A **semantic knowledge base (semantic KB)** indexes your database's *schema*, not its data. It crawls tables, views, columns, and their `COMMENT ON` descriptions, embeds each one with an embedding model, and stores the vectors so they can be searched by meaning.

This lets people and AI agents find the right tables and columns from plain-language questions, even when the words don't match the object names. For example, "customers behind on loan payments" can resolve to `banking.loans.days_past_due` and `banking.loan_payments.payment_status`.

It also crawls foreign keys into **relationships**, so it can answer how tables join as well as which tables exist.

**Why it matters for text-to-SQL:** an agent grounds its SQL in the real schema (actual names, types, comments, and join paths) instead of guessing. This makes the generated queries more accurate.

**Semantic KB vs. vector KB**

| | Vector knowledge base | Semantic knowledge base |
|---|---|---|
| Embeds | Your data (rows, documents) | Your schema (table, view, column metadata) |
| Answers | "What does the content say?" | "Where in the database is this information?" |
| Created with | `aidb.create_pipeline()` | `aidb.create_semantic_kb()` |

In this demo, a semantic KB over the `banking` schema powers an AI agent that discovers the relevant tables, writes SQL, and runs it. Because comments are embedded alongside the structure, well-commented tables give better search results.

## What you will do in this notebook

1. Create a realistic **retail banking schema** (branches, customers, accounts, transactions, loans, and loan payments) with a comment on every table and column.
2. Load **sample data** that includes interesting cases to ask about, such as delinquent loans and flagged transactions.
3. Create the **semantic KB** over that schema.
4. Create a **Claude model** and an **agent** that uses the KB's search tools together with a SQL tool.
5. **Ask questions** in plain English, including a follow-up that depends on the previous answer.

## Prerequisites

- A database with AIDB installed.
- An Anthropic (Claude) API key, which you will be prompted for in the next cell.

## Claude API Key

Run the next cell and paste your Anthropic API key when prompted. `getpass` hides the input, and the key is kept in memory only. It is wrapped in a JSON document (`creds`) that a later cell passes to `aidb.create_model` as the named parameter `:creds`.

In [ ]:
# Provide your Claude API Key
import json
from getpass import getpass
claude_api_key = getpass("Claude API key: ")
creds = json.dumps({"api_key": claude_api_key})

%config SqlMagic.named_parameters = "enabled"

## Connect to the database

Creates a SQLAlchemy engine for the database and attaches the `%sql` / `%%sql` magics to it. Adjust the connection variables if your database is set up differently.

In [ ]:
from sqlalchemy import create_engine
import os, pwd

PGUSER = pwd.getpwuid(os.geteuid()).pw_name
PGPORT="5432"
PGDATABASE = os.environ.get("PGDATABASE", "aidb")

if PGUSER == "postgres":
    PGSOCKET = "/run/postgresql"
else:
    PGSOCKET = "/tmp"

conn = create_engine(
    f"postgresql+psycopg2://{PGUSER}@/{PGDATABASE}?host={PGSOCKET}&port={PGPORT}"
)

%reload_ext sql
%sql conn
%config SqlMagic.named_parameters = "enabled"

## Create sample schema

Financial Services demo: a retail banking schema for the AIDB semantic KB.

The `banking` schema has six related tables:

| Table | What it holds |
|---|---|
| `branches` | Branch locations and the sales region each rolls up to |
| `customers` | Account holders, with segment, credit score, and KYC status |
| `accounts` | Deposit accounts (checking, savings, money market, CDs) and balances |
| `transactions` | Every money movement on an account, including a fraud flag |
| `loans` | Loans issued to customers, with balance and repayment health |
| `loan_payments` | The monthly installment schedule and repayment history |

Two things in this cell matter for the semantic KB:

- **Comments.** Every table and column has a `COMMENT ON` description written in plain business language (for example, that `days_past_due` is "the number of days the oldest unpaid installment is overdue"). The semantic KB embeds these comments, so they are what lets a question about "customers behind on payments" find the right column. Better comments mean better search.
- **Foreign keys.** The tables reference each other (customers to branches, accounts to customers, transactions to accounts, and so on). The KB crawls these into relationships so an agent can work out how to join tables.

In [ ]:
%%sql

DROP SCHEMA IF EXISTS banking CASCADE;
CREATE SCHEMA banking;
COMMENT ON SCHEMA banking IS 'Retail banking data: branches, customers, deposit accounts, account transactions, loans, and loan repayments.';

CREATE TABLE banking.branches (
    branch_id    INT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    branch_name  TEXT NOT NULL,
    city         TEXT NOT NULL,
    state_code   CHAR(2) NOT NULL,
    region       TEXT NOT NULL,
    opened_date  DATE NOT NULL
);

COMMENT ON TABLE  banking.branches IS 'Physical bank branch locations. Each customer and account is serviced by one branch.';
COMMENT ON COLUMN banking.branches.branch_id   IS 'Unique identifier for the branch.';
COMMENT ON COLUMN banking.branches.branch_name IS 'Display name of the branch, e.g. Downtown or Airport.';
COMMENT ON COLUMN banking.branches.city        IS 'City where the branch is located.';
COMMENT ON COLUMN banking.branches.state_code  IS 'Two-letter US state abbreviation of the branch location.';
COMMENT ON COLUMN banking.branches.region      IS 'Sales region the branch rolls up to: Northeast, Southeast, Midwest, or West. Use for regional performance reporting.';
COMMENT ON COLUMN banking.branches.opened_date IS 'Date the branch opened for business.';

CREATE TABLE banking.customers (
    customer_id     INT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    home_branch_id  INT NOT NULL REFERENCES banking.branches(branch_id),
    first_name      TEXT NOT NULL,
    last_name       TEXT NOT NULL,
    email           TEXT NOT NULL,
    date_of_birth   DATE NOT NULL,
    segment         TEXT NOT NULL CHECK (segment IN ('Mass Market','Affluent','Private Banking','Small Business')),
    credit_score    INT CHECK (credit_score BETWEEN 300 AND 850),
    kyc_status      TEXT NOT NULL CHECK (kyc_status IN ('Verified','Pending','Expired')),
    customer_since  DATE NOT NULL
);

COMMENT ON TABLE  banking.customers IS 'Individual and small-business bank customers (the account holders). One row per customer; a customer can own many accounts and loans.';
COMMENT ON COLUMN banking.customers.customer_id    IS 'Unique identifier for the customer.';
COMMENT ON COLUMN banking.customers.home_branch_id IS 'Branch that primarily services the customer. References banking.branches.';
COMMENT ON COLUMN banking.customers.first_name     IS 'Customer given name.';
COMMENT ON COLUMN banking.customers.last_name      IS 'Customer family name.';
COMMENT ON COLUMN banking.customers.email          IS 'Customer contact email address. Unique per customer.';
COMMENT ON COLUMN banking.customers.date_of_birth  IS 'Customer date of birth, used for age-based product eligibility.';
COMMENT ON COLUMN banking.customers.segment        IS 'Relationship tier used for marketing and service level: Mass Market, Affluent, Private Banking, or Small Business.';
COMMENT ON COLUMN banking.customers.credit_score   IS 'Most recent credit bureau (FICO-style) score from 300 to 850. Higher means lower credit risk. NULL if not yet scored.';
COMMENT ON COLUMN banking.customers.kyc_status     IS 'Know-Your-Customer compliance status: Verified, Pending (documents under review), or Expired (needs re-verification).';
COMMENT ON COLUMN banking.customers.customer_since IS 'Date the customer first opened a relationship with the bank.';

CREATE TABLE banking.accounts (
    account_id      INT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    customer_id     INT NOT NULL REFERENCES banking.customers(customer_id),
    branch_id       INT NOT NULL REFERENCES banking.branches(branch_id),
    account_number  TEXT NOT NULL,
    account_type    TEXT NOT NULL CHECK (account_type IN ('Checking','Savings','Money Market','Certificate of Deposit')),
    current_balance NUMERIC(14,2) NOT NULL,
    interest_rate   NUMERIC(5,4) NOT NULL DEFAULT 0,
    status          TEXT NOT NULL CHECK (status IN ('Active','Dormant','Frozen','Closed')),
    opened_date     DATE NOT NULL,
    closed_date     DATE
);

COMMENT ON TABLE  banking.accounts IS 'Deposit accounts (checking, savings, money market, CDs) held by customers. Balances are in USD. Joins to customers and branches; each account has many transactions.';
COMMENT ON COLUMN banking.accounts.account_id      IS 'Unique internal identifier for the account.';
COMMENT ON COLUMN banking.accounts.customer_id     IS 'Owner of the account. References banking.customers.';
COMMENT ON COLUMN banking.accounts.branch_id       IS 'Branch where the account was opened and is serviced. References banking.branches.';
COMMENT ON COLUMN banking.accounts.account_number  IS 'Customer-facing account number. Unique.';
COMMENT ON COLUMN banking.accounts.account_type    IS 'Product type: Checking, Savings, Money Market, or Certificate of Deposit.';
COMMENT ON COLUMN banking.accounts.current_balance IS 'Current ledger balance of the account in USD. Positive means funds on deposit; negative means overdrawn.';
COMMENT ON COLUMN banking.accounts.interest_rate   IS 'Annual percentage yield paid on the balance, as a decimal fraction (0.0425 = 4.25%).';
COMMENT ON COLUMN banking.accounts.status          IS 'Lifecycle state: Active, Dormant (no activity for 12+ months), Frozen (compliance or legal hold), or Closed.';
COMMENT ON COLUMN banking.accounts.opened_date     IS 'Date the account was opened.';
COMMENT ON COLUMN banking.accounts.closed_date     IS 'Date the account was closed. NULL while the account is still open.';

CREATE TABLE banking.transactions (
    transaction_id    BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    account_id        INT NOT NULL REFERENCES banking.accounts(account_id),
    transaction_date  TIMESTAMP NOT NULL,
    transaction_type  TEXT NOT NULL CHECK (transaction_type IN ('Deposit','Withdrawal','Transfer In','Transfer Out','Card Purchase','Wire','Fee','Interest')),
    amount            NUMERIC(14,2) NOT NULL CHECK (amount > 0),
    direction         CHAR(1) NOT NULL CHECK (direction IN ('C','D')),
    channel           TEXT NOT NULL CHECK (channel IN ('Branch','ATM','Online','Mobile','Wire Desk','System')),
    merchant_name     TEXT,
    merchant_category TEXT,
    description       TEXT,
    flagged_fraud     BOOLEAN NOT NULL DEFAULT FALSE
);

COMMENT ON TABLE  banking.transactions IS 'Ledger of every money movement on a deposit account: deposits, withdrawals, transfers, card purchases, wires, fees, and interest. One row per posting; amount is always positive and direction says whether it adds or removes funds.';
COMMENT ON COLUMN banking.transactions.transaction_id    IS 'Unique identifier for the transaction posting.';
COMMENT ON COLUMN banking.transactions.account_id        IS 'Account the transaction posted to. References banking.accounts.';
COMMENT ON COLUMN banking.transactions.transaction_date  IS 'Date and time the transaction posted to the account.';
COMMENT ON COLUMN banking.transactions.transaction_type  IS 'Kind of activity: Deposit, Withdrawal, Transfer In, Transfer Out, Card Purchase, Wire, Fee, or Interest.';
COMMENT ON COLUMN banking.transactions.amount            IS 'Absolute transaction value in USD. Always positive; use direction to determine the sign.';
COMMENT ON COLUMN banking.transactions.direction         IS 'C = credit (money into the account), D = debit (money out of the account).';
COMMENT ON COLUMN banking.transactions.channel           IS 'Where the transaction originated: Branch, ATM, Online, Mobile, Wire Desk, or System (automated, e.g. interest and fees).';
COMMENT ON COLUMN banking.transactions.merchant_name     IS 'Name of the merchant or counterparty for card purchases and wires. NULL for other transaction types.';
COMMENT ON COLUMN banking.transactions.merchant_category IS 'Merchant category such as Groceries, Travel, Dining, Utilities, or Electronics. Use for spending analysis. NULL for non-merchant transactions.';
COMMENT ON COLUMN banking.transactions.description       IS 'Free-text memo describing the transaction as shown on the customer statement.';
COMMENT ON COLUMN banking.transactions.flagged_fraud     IS 'TRUE if the fraud detection system flagged this transaction as suspicious and queued it for review.';

CREATE TABLE banking.loans (
    loan_id             INT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    customer_id         INT NOT NULL REFERENCES banking.customers(customer_id),
    branch_id           INT NOT NULL REFERENCES banking.branches(branch_id),
    loan_type           TEXT NOT NULL CHECK (loan_type IN ('Mortgage','Auto','Personal','Home Equity','Small Business')),
    principal_amount    NUMERIC(14,2) NOT NULL,
    outstanding_balance NUMERIC(14,2) NOT NULL,
    interest_rate       NUMERIC(5,4) NOT NULL,
    term_months         INT NOT NULL,
    origination_date    DATE NOT NULL,
    status              TEXT NOT NULL CHECK (status IN ('Current','Delinquent','Default','Paid Off')),
    days_past_due       INT NOT NULL DEFAULT 0
);

COMMENT ON TABLE  banking.loans IS 'Loans the bank has issued to customers: mortgages, auto, personal, home equity, and small business. Tracks original amount, remaining balance, and repayment health. Each loan has many scheduled payments in loan_payments.';
COMMENT ON COLUMN banking.loans.loan_id             IS 'Unique identifier for the loan.';
COMMENT ON COLUMN banking.loans.customer_id         IS 'Borrower. References banking.customers.';
COMMENT ON COLUMN banking.loans.branch_id           IS 'Branch that originated the loan. References banking.branches.';
COMMENT ON COLUMN banking.loans.loan_type           IS 'Loan product: Mortgage, Auto, Personal, Home Equity, or Small Business.';
COMMENT ON COLUMN banking.loans.principal_amount    IS 'Original amount borrowed in USD at origination.';
COMMENT ON COLUMN banking.loans.outstanding_balance IS 'Principal still owed in USD. Sum this for total loan portfolio exposure.';
COMMENT ON COLUMN banking.loans.interest_rate       IS 'Annual interest rate charged to the borrower, as a decimal fraction (0.0650 = 6.50%).';
COMMENT ON COLUMN banking.loans.term_months         IS 'Contractual length of the loan in months.';
COMMENT ON COLUMN banking.loans.origination_date    IS 'Date the loan was funded.';
COMMENT ON COLUMN banking.loans.status              IS 'Repayment health: Current (on schedule), Delinquent (late), Default (written off or in collections), or Paid Off.';
COMMENT ON COLUMN banking.loans.days_past_due       IS 'Number of days the oldest unpaid installment is overdue. 0 when current. Common delinquency buckets are 30, 60, and 90+ days.';

CREATE TABLE banking.loan_payments (
    payment_id        INT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    loan_id           INT NOT NULL REFERENCES banking.loans(loan_id),
    funding_account_id INT REFERENCES banking.accounts(account_id),
    due_date          DATE NOT NULL,
    paid_date         DATE,
    amount_due        NUMERIC(12,2) NOT NULL,
    amount_paid       NUMERIC(12,2) NOT NULL DEFAULT 0,
    payment_status    TEXT NOT NULL CHECK (payment_status IN ('Paid','Late','Missed','Scheduled'))
);

COMMENT ON TABLE  banking.loan_payments IS 'Installment schedule and repayment history for loans. One row per monthly payment due, showing whether and when the borrower paid.';
COMMENT ON COLUMN banking.loan_payments.payment_id         IS 'Unique identifier for the loan installment.';
COMMENT ON COLUMN banking.loan_payments.loan_id            IS 'Loan this installment belongs to. References banking.loans.';
COMMENT ON COLUMN banking.loan_payments.funding_account_id IS 'Deposit account the payment was drawn from. References banking.accounts. NULL if paid by cash, check, or an outside bank.';
COMMENT ON COLUMN banking.loan_payments.due_date           IS 'Date the installment was due.';
COMMENT ON COLUMN banking.loan_payments.paid_date          IS 'Date the payment was received. NULL if not yet paid.';
COMMENT ON COLUMN banking.loan_payments.amount_due         IS 'Scheduled installment amount in USD (principal plus interest).';
COMMENT ON COLUMN banking.loan_payments.amount_paid        IS 'Amount actually received in USD. Less than amount_due indicates a partial payment.';
COMMENT ON COLUMN banking.loan_payments.payment_status     IS 'Paid (on time), Late (paid after due date), Missed (past due and unpaid), or Scheduled (due in the future).';


## Load sample data

Inserts a small dataset: 7 branches, 15 customers, 25 accounts, around 50 transactions, 15 loans, and about 30 loan payments.

The data is small but deliberately varied so there are interesting questions to ask: an overdrawn checking account, frozen, dormant, and closed accounts, customers with pending or expired KYC, fraud-flagged transactions, and loans that are delinquent, in default, or paid off, with missed and late payments.

In [ ]:
%%sql

INSERT INTO banking.branches (branch_name, city, state_code, region, opened_date) VALUES
 ('Downtown',        'New York',     'NY', 'Northeast', '1998-03-15'),
 ('Back Bay',        'Boston',       'MA', 'Northeast', '2005-09-01'),
 ('Peachtree',       'Atlanta',      'GA', 'Southeast', '2001-06-20'),
 ('Brickell',        'Miami',        'FL', 'Southeast', '2012-11-05'),
 ('The Loop',        'Chicago',      'IL', 'Midwest',   '1995-01-10'),
 ('Silicon Valley',  'San Jose',     'CA', 'West',      '2008-04-22'),
 ('Pioneer Square',  'Seattle',      'WA', 'West',      '2015-08-17');

INSERT INTO banking.customers (home_branch_id, first_name, last_name, email, date_of_birth, segment, credit_score, kyc_status, customer_since) VALUES
 (1, 'Olivia',   'Bennett',   'olivia.bennett@example.com',   '1985-04-12', 'Affluent',        782, 'Verified', '2014-02-10'),
 (1, 'Marcus',   'Reyes',     'marcus.reyes@example.com',     '1979-11-03', 'Private Banking', 811, 'Verified', '2009-07-21'),
 (2, 'Priya',    'Nair',      'priya.nair@example.com',       '1992-08-27', 'Mass Market',     701, 'Verified', '2019-05-14'),
 (2, 'Daniel',   'Kowalski',  'daniel.kowalski@example.com',  '1968-01-19', 'Small Business',  665, 'Verified', '2011-10-02'),
 (3, 'Tasha',    'Williams',  'tasha.williams@example.com',   '1990-06-30', 'Mass Market',     634, 'Pending',  '2022-03-08'),
 (3, 'Robert',   'Chen',     'robert.chen@example.com',       '1974-12-09', 'Affluent',        745, 'Verified', '2013-01-25'),
 (4, 'Sofia',    'Martinez',  'sofia.martinez@example.com',   '1988-09-15', 'Mass Market',     688, 'Verified', '2020-11-30'),
 (4, 'Jamal',    'Thompson',  'jamal.thompson@example.com',   '1983-02-22', 'Small Business',  598, 'Expired',  '2016-06-12'),
 (5, 'Eleanor',  'Fitzgerald','eleanor.fitz@example.com',     '1956-05-05', 'Private Banking', 829, 'Verified', '2002-09-17'),
 (5, 'Kevin',    'Okafor',    'kevin.okafor@example.com',     '1995-07-18', 'Mass Market',     672, 'Verified', '2021-04-19'),
 (6, 'Mei',      'Lin',       'mei.lin@example.com',          '1987-03-11', 'Affluent',        768, 'Verified', '2015-12-01'),
 (6, 'Andre',    'Dubois',    'andre.dubois@example.com',     '1971-10-28', 'Small Business',  710, 'Verified', '2010-08-09'),
 (7, 'Hannah',   'Schmidt',   'hannah.schmidt@example.com',   '1999-01-07', 'Mass Market',     655, 'Verified', '2023-02-14'),
 (7, 'Victor',   'Alvarez',   'victor.alvarez@example.com',   '1963-06-14', 'Affluent',        790, 'Verified', '2007-05-23'),
 (1, 'Grace',    'Park',      'grace.park@example.com',       '1994-11-21', 'Mass Market',     720, 'Verified', '2018-09-06');

INSERT INTO banking.accounts (customer_id, branch_id, account_number, account_type, current_balance, interest_rate, status, opened_date, closed_date) VALUES
 (1,  1, 'CHK-100001', 'Checking',               18450.22, 0.0010, 'Active',  '2014-02-10', NULL),
 (1,  1, 'SAV-100002', 'Savings',                62000.00, 0.0425, 'Active',  '2014-02-10', NULL),
 (2,  1, 'CHK-100003', 'Checking',              245100.75, 0.0025, 'Active',  '2009-07-21', NULL),
 (2,  1, 'MMA-100004', 'Money Market',          850000.00, 0.0475, 'Active',  '2012-01-15', NULL),
 (2,  1, 'CDA-100005', 'Certificate of Deposit',500000.00, 0.0525, 'Active',  '2024-03-01', NULL),
 (3,  2, 'CHK-100006', 'Checking',               2310.48, 0.0000, 'Active',  '2019-05-14', NULL),
 (3,  2, 'SAV-100007', 'Savings',                5400.00, 0.0400, 'Active',  '2019-06-01', NULL),
 (4,  2, 'CHK-100008', 'Checking',              73420.10, 0.0010, 'Active',  '2011-10-02', NULL),
 (5,  3, 'CHK-100009', 'Checking',                -85.40, 0.0000, 'Active',  '2022-03-08', NULL),
 (6,  3, 'CHK-100010', 'Checking',               9875.33, 0.0010, 'Active',  '2013-01-25', NULL),
 (6,  3, 'SAV-100011', 'Savings',               41200.00, 0.0425, 'Active',  '2013-01-25', NULL),
 (7,  4, 'CHK-100012', 'Checking',               3120.67, 0.0000, 'Active',  '2020-11-30', NULL),
 (8,  4, 'CHK-100013', 'Checking',               1204.90, 0.0000, 'Frozen',  '2016-06-12', NULL),
 (9,  5, 'CHK-100014', 'Checking',             512340.00, 0.0025, 'Active',  '2002-09-17', NULL),
 (9,  5, 'MMA-100015', 'Money Market',        1250000.00, 0.0475, 'Active',  '2005-04-04', NULL),
 (10, 5, 'CHK-100016', 'Checking',               1875.12, 0.0000, 'Active',  '2021-04-19', NULL),
 (10, 5, 'SAV-100017', 'Savings',                 300.00, 0.0400, 'Dormant', '2021-04-19', NULL),
 (11, 6, 'CHK-100018', 'Checking',              22980.55, 0.0010, 'Active',  '2015-12-01', NULL),
 (11, 6, 'SAV-100019', 'Savings',               88000.00, 0.0425, 'Active',  '2015-12-01', NULL),
 (12, 6, 'CHK-100020', 'Checking',             134500.00, 0.0010, 'Active',  '2010-08-09', NULL),
 (13, 7, 'CHK-100021', 'Checking',               1560.00, 0.0000, 'Active',  '2023-02-14', NULL),
 (14, 7, 'CHK-100022', 'Checking',              67300.90, 0.0025, 'Active',  '2007-05-23', NULL),
 (14, 7, 'CDA-100023', 'Certificate of Deposit',150000.00, 0.0500, 'Active',  '2025-01-10', NULL),
 (15, 1, 'CHK-100024', 'Checking',               4210.30, 0.0010, 'Active',  '2018-09-06', NULL),
 (15, 1, 'SAV-100025', 'Savings',                    0.00, 0.0400, 'Closed',  '2018-09-06', '2025-06-30');

-- Transactions: a mix of deposits, card spend, transfers, fees, wires, and flagged items
INSERT INTO banking.transactions (account_id, transaction_date, transaction_type, amount, direction, channel, merchant_name, merchant_category, description, flagged_fraud) VALUES
 -- Olivia checking (1)
 (1, '2026-09-01 08:00', 'Deposit',       6200.00, 'C', 'System',  NULL,                 NULL,          'Payroll direct deposit - Acme Corp', FALSE),
 (1, '2026-09-03 12:41', 'Card Purchase',   86.27, 'D', 'Mobile',  'Whole Foods Market', 'Groceries',   'WHOLE FOODS #1042 NEW YORK NY', FALSE),
 (1, '2026-09-05 19:22', 'Card Purchase',  142.90, 'D', 'Online',  'Delta Air Lines',    'Travel',      'DELTA AIR LINES ATLANTA', FALSE),
 (1, '2026-09-10 09:15', 'Transfer Out', 1000.00, 'D', 'Mobile',  NULL,                 NULL,          'Transfer to savings SAV-100002', FALSE),
 (1, '2026-09-14 20:05', 'Card Purchase',   64.10, 'D', 'Online',  'The Capital Grille', 'Dining',      'CAPITAL GRILLE NEW YORK', FALSE),
 (1, '2026-09-30 23:59', 'Interest',         1.52, 'C', 'System',  NULL,                 NULL,          'Monthly interest credit', FALSE),
 -- Olivia savings (2)
 (2, '2026-09-10 09:15', 'Transfer In',  1000.00, 'C', 'Mobile',  NULL,                 NULL,          'Transfer from checking CHK-100001', FALSE),
 (2, '2026-09-30 23:59', 'Interest',       219.80, 'C', 'System',  NULL,                 NULL,          'Monthly interest credit', FALSE),
 -- Marcus checking (3)
 (3, '2026-09-02 10:30', 'Wire',        75000.00, 'D', 'Wire Desk','Sotheby''s',        'Luxury Goods','Wire to Sotheby''s auction house', FALSE),
 (3, '2026-09-08 14:00', 'Deposit',    120000.00, 'C', 'Branch',  NULL,                 NULL,          'Deposit of dividend check', FALSE),
 (3, '2026-09-18 21:47', 'Card Purchase', 9800.00,'D', 'Online',  'Apple Store',        'Electronics', 'APPLE.COM/US', TRUE),
 -- Priya checking (6)
 (6, '2026-09-01 08:00', 'Deposit',      2900.00, 'C', 'System',  NULL,                 NULL,          'Payroll direct deposit - Mass General', FALSE),
 (6, '2026-09-04 18:12', 'Card Purchase',  212.45, 'D', 'Mobile',  'Trader Joe''s',      'Groceries',   'TRADER JOE''S #502 BOSTON', FALSE),
 (6, '2026-09-06 07:50', 'Card Purchase',   5.75, 'D', 'Mobile',  'Starbucks',          'Dining',      'STARBUCKS BOSTON MA', FALSE),
 (6, '2026-09-15 12:00', 'Card Purchase', 1450.00, 'D', 'Online',  'Boston Property Mgmt','Housing',    'Rent payment', FALSE),
 (6, '2026-09-20 11:00', 'Fee',            12.00, 'D', 'System',  NULL,                 NULL,          'Monthly maintenance fee', FALSE),
 -- Daniel business checking (8)
 (8, '2026-09-03 15:20', 'Deposit',      18250.00, 'C', 'Branch',  NULL,                 NULL,          'Customer receipts deposit', FALSE),
 (8, '2026-09-12 10:00', 'Wire',         22000.00, 'D', 'Wire Desk','Midwest Supply Co', 'Supplies',   'Wire to supplier invoice 8841', FALSE),
 (8, '2026-09-25 16:44', 'Card Purchase', 640.18, 'D', 'Online',  'Staples',            'Office',      'STAPLES.COM', FALSE),
 -- Tasha overdrawn checking (9)
 (9, '2026-09-02 08:00', 'Deposit',      1800.00, 'C', 'System',  NULL,                 NULL,          'Payroll direct deposit', FALSE),
 (9, '2026-09-05 13:00', 'Card Purchase',  945.60, 'D', 'Online',  'Best Buy',           'Electronics', 'BESTBUY.COM', FALSE),
 (9, '2026-09-18 17:30', 'Withdrawal',    1000.00, 'D', 'ATM',     NULL,                 NULL,          'ATM cash withdrawal', FALSE),
 (9, '2026-09-18 17:31', 'Fee',            35.00, 'D', 'System',  NULL,                 NULL,          'Overdraft fee', FALSE),
 -- Robert checking (10)
 (10,'2026-09-01 08:00', 'Deposit',      5400.00, 'C', 'System',  NULL,                 NULL,          'Payroll direct deposit - Delta', FALSE),
 (10,'2026-09-09 19:45', 'Card Purchase',  312.88, 'D', 'Mobile',  'Kroger',             'Groceries',   'KROGER #221 ATLANTA', FALSE),
 (10,'2026-09-21 22:10', 'Card Purchase', 2150.00, 'D', 'Online',  'Marriott Hotels',    'Travel',      'MARRIOTT RESERVATION', FALSE),
 -- Sofia checking (12)
 (12,'2026-09-01 08:00', 'Deposit',      2400.00, 'C', 'System',  NULL,                 NULL,          'Payroll direct deposit', FALSE),
 (12,'2026-09-07 20:30', 'Card Purchase',   78.30, 'D', 'Mobile',  'Publix',             'Groceries',   'PUBLIX #118 MIAMI', FALSE),
 (12,'2026-09-13 23:15', 'Card Purchase',  499.99, 'D', 'Online',  'Unknown Merchant',   'Electronics', 'ONLINE PURCHASE INTL', TRUE),
 -- Eleanor checking (14)
 (14,'2026-09-05 11:00', 'Deposit',      90000.00, 'C', 'Wire Desk',NULL,               NULL,          'Incoming wire - trust distribution', FALSE),
 (14,'2026-09-16 13:30', 'Card Purchase', 4200.00, 'D', 'Branch',  'Neiman Marcus',      'Retail',      'NEIMAN MARCUS CHICAGO', FALSE),
 (14,'2026-09-24 09:00', 'Transfer Out', 50000.00, 'D', 'Online',  NULL,                 NULL,          'Transfer to money market MMA-100015', FALSE),
 -- Kevin checking (16)
 (16,'2026-09-01 08:00', 'Deposit',      2100.00, 'C', 'System',  NULL,                 NULL,          'Payroll direct deposit', FALSE),
 (16,'2026-09-11 18:00', 'Card Purchase',  156.20, 'D', 'Mobile',  'Jewel-Osco',         'Groceries',   'JEWEL-OSCO CHICAGO', FALSE),
 (16,'2026-09-19 21:15', 'Card Purchase',   42.75, 'D', 'Mobile',  'Uber',               'Transportation','UBER TRIP', FALSE),
 -- Mei checking (18)
 (18,'2026-09-01 08:00', 'Deposit',      9800.00, 'C', 'System',  NULL,                 NULL,          'Payroll direct deposit - TechCo', FALSE),
 (18,'2026-09-14 12:10', 'Card Purchase',  420.00, 'D', 'Mobile',  'Costco',             'Groceries',   'COSTCO WHOLESALE SAN JOSE', FALSE),
 (18,'2026-09-22 15:35', 'Card Purchase', 1899.00, 'D', 'Online',  'Amazon',             'Electronics', 'AMAZON.COM', FALSE),
 -- Andre business checking (20)
 (20,'2026-09-04 14:00', 'Deposit',      41200.00, 'C', 'Branch',  NULL,                 NULL,          'Client payments deposit', FALSE),
 (20,'2026-09-15 10:00', 'Wire',         30000.00, 'D', 'Wire Desk','Bay Area Properties','Housing',    'Wire for office lease', FALSE),
 -- Hannah checking (21)
 (21,'2026-09-01 08:00', 'Deposit',      1950.00, 'C', 'System',  NULL,                 NULL,          'Payroll direct deposit', FALSE),
 (21,'2026-09-08 17:55', 'Card Purchase',  118.40, 'D', 'Mobile',  'Safeway',            'Groceries',   'SAFEWAY SEATTLE', FALSE),
 (21,'2026-09-27 01:12', 'Card Purchase', 2300.00, 'D', 'Online',  'Overseas Gaming Ltd','Gambling',    'INTL ONLINE GAMING', TRUE),
 -- Victor checking (22)
 (22,'2026-09-03 09:30', 'Deposit',      15000.00, 'C', 'Branch',  NULL,                 NULL,          'Consulting income deposit', FALSE),
 (22,'2026-09-17 19:00', 'Card Purchase',  860.00, 'D', 'Online',  'Alaska Airlines',    'Travel',      'ALASKA AIR SEATTLE', FALSE),
 -- Grace checking (24)
 (24,'2026-09-01 08:00', 'Deposit',      3300.00, 'C', 'System',  NULL,                 NULL,          'Payroll direct deposit', FALSE),
 (24,'2026-09-12 08:20', 'Card Purchase',   14.60, 'D', 'Mobile',  'Starbucks',          'Dining',      'STARBUCKS NEW YORK NY', FALSE),
 (24,'2026-09-28 13:00', 'Card Purchase',  275.00, 'D', 'Online',  'Con Edison',         'Utilities',   'CON ED UTILITY PAYMENT', FALSE);

INSERT INTO banking.loans (customer_id, branch_id, loan_type, principal_amount, outstanding_balance, interest_rate, term_months, origination_date, status, days_past_due) VALUES
 (1,  1, 'Mortgage',       640000.00, 598200.00, 0.0625, 360, '2021-08-15', 'Current',    0),
 (2,  1, 'Home Equity',    250000.00, 212500.00, 0.0710, 180, '2022-05-20', 'Current',    0),
 (3,  2, 'Auto',            28000.00,  19450.00, 0.0599,  60, '2023-02-10', 'Current',    0),
 (4,  2, 'Small Business', 150000.00, 118000.00, 0.0825,  84, '2022-10-01', 'Current',    0),
 (5,  3, 'Personal',        12000.00,   9800.00, 0.1299,  48, '2024-06-12', 'Delinquent', 45),
 (6,  3, 'Mortgage',       410000.00, 372400.00, 0.0575, 360, '2020-03-30', 'Current',    0),
 (7,  4, 'Auto',            34000.00,  27100.00, 0.0699,  72, '2024-01-18', 'Current',    0),
 (8,  4, 'Small Business',  90000.00,  81500.00, 0.0990,  60, '2024-09-05', 'Default',  120),
 (10, 5, 'Personal',         8000.00,   6200.00, 0.1150,  36, '2024-11-22', 'Delinquent', 32),
 (11, 6, 'Mortgage',       920000.00, 861000.00, 0.0650, 360, '2022-01-10', 'Current',    0),
 (12, 6, 'Small Business', 220000.00, 164000.00, 0.0775,  96, '2021-06-14', 'Current',    0),
 (13, 7, 'Auto',            22000.00,  20100.00, 0.0899,  60, '2025-03-03', 'Delinquent', 35),
 (14, 7, 'Mortgage',       530000.00,       0.00, 0.0450, 180, '2008-04-01', 'Paid Off',   0),
 (9,  5, 'Home Equity',    300000.00, 255000.00, 0.0680, 240, '2021-11-09', 'Current',    0),
 (15, 1, 'Personal',         5000.00,       0.00, 0.1099,  24, '2023-06-01', 'Paid Off',   0);

INSERT INTO banking.loan_payments (loan_id, funding_account_id, due_date, paid_date, amount_due, amount_paid, payment_status) VALUES
 -- Loan 1 (Olivia mortgage)
 (1, 1, '2026-07-01', '2026-06-29', 3940.00, 3940.00, 'Paid'),
 (1, 1, '2026-08-01', '2026-07-31', 3940.00, 3940.00, 'Paid'),
 (1, 1, '2026-09-01', '2026-09-01', 3940.00, 3940.00, 'Paid'),
 (1, 1, '2026-10-01', '2026-10-01', 3940.00, 3940.00, 'Paid'),
 (1, 1, current_date + 30, NULL,         3940.00,    0.00, 'Scheduled'),
 -- Loan 3 (Priya auto)
 (3, 6, '2026-08-05', '2026-08-04',  540.00,  540.00, 'Paid'),
 (3, 6, '2026-09-05', '2026-09-05',  540.00,  540.00, 'Paid'),
 (3, 6, '2026-10-05', '2026-10-05',  540.00,  540.00, 'Paid'),
 (3, 6, current_date + 30, NULL,          540.00,    0.00, 'Scheduled'),
 -- Loan 4 (Daniel small business)
 (4, 8, '2026-08-01', '2026-08-01', 2350.00, 2350.00, 'Paid'),
 (4, 8, '2026-09-01', '2026-09-03', 2350.00, 2350.00, 'Late'),
 (4, 8, '2026-10-01', '2026-10-01', 2350.00, 2350.00, 'Paid'),
 -- Loan 5 (Tasha personal, delinquent)
 (5, 9, '2026-08-12', '2026-08-30',  320.00,  320.00, 'Late'),
 (5, 9, '2026-09-12', NULL,          320.00,    0.00, 'Missed'),
 (5, 9, '2026-10-12', NULL,          320.00,    0.00, 'Missed'),
 -- Loan 8 (Jamal small business, default)
 (8, 13,'2026-06-05', NULL,         1910.00,    0.00, 'Missed'),
 (8, 13,'2026-07-05', NULL,         1910.00,    0.00, 'Missed'),
 (8, 13,'2026-08-05', NULL,         1910.00,    0.00, 'Missed'),
 (8, 13,'2026-09-05', NULL,         1910.00,    0.00, 'Missed'),
 -- Loan 9 (Kevin personal, delinquent)
 (9, 16,'2026-08-22', '2026-08-22',  264.00,  264.00, 'Paid'),
 (9, 16,'2026-09-22', NULL,          264.00,    0.00, 'Missed'),
 (9, 16,'2026-10-22', NULL,          264.00,  100.00, 'Late'),
 -- Loan 10 (Mei mortgage)
 (10,18,'2026-09-01', '2026-09-01', 5815.00, 5815.00, 'Paid'),
 (10,18,'2026-10-01', '2026-10-01', 5815.00, 5815.00, 'Paid'),
 (10,18,current_date + 30, NULL,         5815.00,    0.00, 'Scheduled'),
 -- Loan 12 (Hannah auto, delinquent)
 (12,21,'2026-08-15', '2026-09-01',  455.00,  455.00, 'Late'),
 (12,21,'2026-09-15', NULL,          455.00,    0.00, 'Missed'),
 (12,21,'2026-10-15', NULL,          455.00,    0.00, 'Missed');


## Create the Model

This uses Claude Sonnet 5. The model is used for the reasoning when you ask questions via an agent: deciding which tools to call, writing the SQL, and explaining the result. The embedding model used by the KB is separate and is configured in the previous step.

In [ ]:
%%sql
DO $do$
DECLARE
    v_name text := 'claude_model_semantic_kb';
BEGIN
    IF EXISTS (SELECT NULL FROM aidb.models WHERE name = v_name) THEN
        PERFORM aidb.delete_model(v_name);
    END IF;

    PERFORM aidb.create_model(
        name        => v_name,
        provider    => 'anthropic_messages',
        config      => aidb.anthropic_messages_config(model => 'claude-sonnet-5'),
        credentials => CAST(:creds AS JSONB),
        replace_credentials => true
    );
END
$do$;

## Create Semantic KB

`aidb.create_semantic_kb` crawls the listed schemas and builds the searchable index.

- `name`: the identifier agents use to refer to this KB (`banking_semantic_kb`).
- `model`: the embedding model (`bge-small-en-v1.5-f16`) used to turn table, column, and comment text into vectors.
- `schemas`: which schemas to index. Here, only `banking`.
- `auto_processing => 'Live'`: keeps the KB current. DDL triggers re-crawl affected relations as soon as the schema changes, so adding a column or editing a comment is reflected without any manual step.

The first statement removes an existing KB with the same name so the cell can be re-run. You can list the KBs in your database at any time with `aidb.list_semantic_kbs()`.

In [ ]:
%%sql
DO $do$
DECLARE
    v_name text := 'banking_semantic_kb';
BEGIN
    IF EXISTS (SELECT NULL FROM aidb.list_semantic_kbs() WHERE name = v_name) THEN
        PERFORM aidb.delete_semantic_kb(v_name);
    END IF;

    PERFORM aidb.create_semantic_kb(
        name    => v_name,
        model   => 'bge-small-en-v1.5-f16',
        schemas => ARRAY['banking'],
        auto_processing => 'Live'
    );
END
$do$;

## Create the Agent

Using the Claude model just created above, create an agent.

Notice that **five tools** are used here. Together they let the agent find the right tables, work out how they connect, and then query them:

| Tool | Role |
|---|---|
| `semantic_kb_search` | Find relevant tables and columns by meaning |
| `search_by_comment` | Search using the descriptions in object comments |
| `list_relationships` | See how tables are linked by foreign keys |
| `find_join_path` | Work out the route for joining tables that are not directly related |
| `run_sql_query` | Run the read-only query the agent writes |

The `instructions` spell out a process for the agent to follow: search the KB, find a join path if needed, write one read-only `SELECT` using only objects it found, run it, and answer briefly while showing the SQL. They also include the key guardrail, *never guess table or column names*. `max_iterations => 10` caps how many tool-calling steps the agent can take for a single question.

In [ ]:
%%sql
DO $do$
DECLARE
    v_name text := 'demo_agent_banking_semantic_kb';
BEGIN
    IF EXISTS (SELECT NULL FROM aidb.agents WHERE name = v_name) THEN
        PERFORM aidb.delete_agent(v_name, force => true);
    END IF;

    PERFORM aidb.create_agent(
      name         => v_name,
      instructions => $$You answer questions about a retail banking database.
    All data lives in the "banking" schema (branches, customers, accounts, transactions, loans, loan_payments). Ignore all other schemas.
    The knowledge base is always kb_name => 'banking_semantic_kb'. Pass it on every semantic_kb_search, search_by_comment, list_relationships and find_join_path call.
    
    Process:
    1. Call semantic_kb_search to find the relevant tables and columns. Reuse names you already found earlier in this conversation instead of searching again. To list the available tables, use entity_types => ['Table'].
    2. If the tables are not directly related, call find_join_path to get the join route.
    3. Write one read-only SELECT using only objects you found. Always schema-qualify tables, for example banking.customers.
    4. Run it with run_sql_query. If it fails on an unknown column, query information_schema.columns for that table. Do not guess.
    5. Answer briefly and show the SQL you ran.
    
    Never guess table or column names. If the schema does not contain what is needed, say so.$$,
      model        => 'claude_model_semantic_kb',
      tools        => ARRAY['semantic_kb_search','search_by_comment','list_relationships','find_join_path','run_sql_query'],
      max_iterations => 10
    );
END
$do$;

## Ask the Agent a Question

The output contains the message and a `conversation_id`. We use a DataFrame so that we can retrieve the message and capture the `conversation_id` for a follow-up.

The question is *"Which customers have delinquent loans and what is their total outstanding balance?"*. Nothing in it names a table. The agent has to discover that the answer lives in `banking.loans` and `banking.customers`, work out the join, and filter on loan status.

With this sample data you should see three borrowers with loans in the `Delinquent` status (Tasha Williams, Kevin Okafor, and Hannah Schmidt) with a combined outstanding balance of $36,100. One more loan is in `Default`, which is a separate status, so whether it appears depends on how the agent interprets the word "delinquent".

In [ ]:
from sqlalchemy import text
from IPython.display import Markdown, display
import pandas as pd

def ask(agent, question, conversation_id=None):
    """Ask the agent, render the answer, return (message, conversation_id).
    Pass a conversation_id from an earlier call to ask a follow-up."""
    with conn.begin() as c:
        df = pd.read_sql(
            text("""SELECT * FROM aidb.agent_converse(
                        CAST(:agent AS text), CAST(:q AS text),
                        conversation_id => CAST(:cid AS text))"""),
            c, params={"agent": agent, "q": question, "cid": conversation_id})
    row = df.iloc[0]
    if row["error"]:
        display(Markdown(f"**Agent error:** {row['error']}"))
    else:
        display(Markdown(row["message"]))
    # On an error the function returns a NULL id, so keep the one passed in
    new_id = row["conversation_id"]
    return row["message"], (conversation_id if pd.isna(new_id) else new_id)

AGENT = "demo_agent_banking_semantic_kb"
_, conversation_id = ask(AGENT, "Which customers have delinquent loans and what is their total outstanding balance?")

## Follow-up question using the conversation_id

Now that we have an answer, ask a follow-up. Passing the `conversation_id` gives the agent the earlier messages as context.

Notice how the LLM has to figure out what I mean by "current one"? The question never says current *what*. The agent has to resolve it from the conversation and from the schema (loans have a status that can be `Current`). If it reads it as loans with a status of `Current`, the largest outstanding balance in this data is Mei Lin's mortgage at $861,000.

In [ ]:
_, conversation_id = ask(AGENT, "Which current one has the largest balance?", conversation_id)

## Try Agent Converse

Try out the Agent Converse app with these searches. You can also run any of them from a cell with `aidb.agent_converse('demo_agent_banking_semantic_kb', '...')`.

- Which branches hold the highest total deposit balances?
- Which accounts are overdrawn, and who owns them?
- Show customers whose KYC status is pending or expired, with their account balances.
- Which transactions were flagged as fraud, and which customers do they belong to?
- What is the total outstanding loan balance by loan type, and by region?
- Which customers have missed loan payments, and which accounts fund those payments?

**Experiment:** edit a table or column comment with `COMMENT ON ...`. Because the KB is `Live`, it re-crawls automatically, and you can watch the change affect which tables the agent finds.